# 🛡️ V9 Bulletproof Scraper

**Mode:** Process Isolation

**Why this works:**
Jupyter blocks Sync Playwright because it has its own Async Loop running. 
We fix this by writing the simplified V9 logic to a script (`v9_script.py`) and running it as a **fresh process**. 
NO more `NotImplementedError`. NO more `loop` errors.

### Instructions
1.  **Run Cell 1** (Setup).
2.  **Run Cell 2** (Creates Script).
3.  **Run Cell 3** (Executes Script).

In [ ]:
# CELL 1: SETUP
!pip install playwright pandas feedparser trafilatura playwright-stealth tqdm requests
!playwright install chromium

In [ ]:
%%writefile v9_script.py
import sys, os, time, json, hashlib, re, sqlite3
import requests, feedparser, trafilatura
import pandas as pd
from urllib.parse import urljoin, urlparse
from datetime import datetime
from tqdm import tqdm

# SYNC PLAYWRIGHT (Safe in this script)
from playwright.sync_api import sync_playwright
try: from playwright_stealth import stealth_sync
except: 
    def stealth_sync(p): p.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# --- CONFIG ---
SOURCES_FILE = "sources.json"
OUTPUT_CSV = "scraped_news_v9.csv"

def load_sources():
    try: 
        with open(SOURCES_FILE) as f: return json.load(f).get('sources', [])
    except: return []

def save_csv(articles):
    if not articles: return
    df = pd.DataFrame(articles)
    hdr = not os.path.exists(OUTPUT_CSV)
    df.to_csv(OUTPUT_CSV, mode='a', header=hdr, index=False, encoding='utf-8')

def get_seen_hashes():
    if os.path.exists(OUTPUT_CSV):
        try: return set(pd.read_csv(OUTPUT_CSV)['url_hash'].tolist())
        except: pass
    return set()

def make_article(source, title, url, content, date, method):
    return {
        "source": source, "headline": (title or "Unknown").strip(),
        "url": url, "published": date,
        "content_snippet": (content or "")[:200].strip(),
        "full_content": (content or "").strip(),
        "url_hash": hashlib.md5(url.encode()).hexdigest(),
        "method": method, "scraped_at": datetime.now().isoformat()
    }

# --- LOGIC ---
def fetch_rss(source, url, seen):
    found = []
    feeds = [urljoin(url, p) for p in ["/feed", "/rss", "/rss.xml"]]
    try:
        r = requests.get(url, timeout=5)
        m = re.search(r'href=\"(.*?)\"[^>]*type=\"application/rss[+]xml\"', r.text)
        if m: feeds.insert(0, urljoin(url, m.group(1)))
    except: pass

    for f in feeds:
        try:
            d = feedparser.parse(f)
            for e in d.entries[:5]:
                if hashlib.md5(e.link.encode()).hexdigest() in seen: continue
                txt = ""
                try: 
                    b = trafilatura.bare_extraction(requests.get(e.link, timeout=5).text)
                    if b: txt = b['text']
                except: pass
                if len(txt) > 50:
                    found.append(make_article(source, e.title, e.link, txt, "", "rss"))
            if found: break
        except: continue
    return found

def run_main():
    sources = load_sources()
    seen = get_seen_hashes()
    print(f"🚀 Starting V9 (Process Isolated)... {len(sources)} sources.")
    
    with sync_playwright() as p:
        print("🌐 Browser Launched.")
        browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
        
        for s in tqdm(sources):
            # 1. RSS
            arts = fetch_rss(s['name'], s['url'], seen)
            
            # 2. Browser
            if not arts:
                # Browser Logic Inline
                try:
                    # tqdm.write(f"   🔎 Scanning {s['name']}...")
                    page = browser.new_page()
                    stealth_sync(page)
                    page.goto(s['url'], timeout=15000)
                    links = page.evaluate("() => Array.from(document.querySelectorAll('a')).map(a => ({href: a.href, text: a.innerText})).filter(a => a.href.startsWith('http') && a.text.length > 20)")
                    
                    domain = urlparse(s['url']).netloc.replace('www.', '')
                    count = 0
                    for l in links:
                        if count >= 4: break
                        if domain not in l['href']: continue
                        if hashlib.md5(l['href'].encode()).hexdigest() in seen: continue
                        
                        try:
                            page.goto(l['href'], timeout=10000)
                            b = trafilatura.bare_extraction(page.content(), url=l['href'])
                            if b and len(b.get('text', '')) > 200:
                                arts.append(make_article(s['name'], b.get('title') or l['text'], l['href'], b['text'], "", "browser"))
                                count += 1
                        except: continue
                    page.close()
                except: pass
            
            if arts:
                save_csv(arts)
                for a in arts: seen.add(a['url_hash'])
        
        browser.close()
    print("✨ Done.")

if __name__ == "__main__":
    run_main()

In [ ]:
# CELL 3: RUN IT (The Magic Fix)
!python v9_script.py

In [ ]:
# CELL 4: VIEW DATA
import pandas as pd, os
if os.path.exists("scraped_news_v9.csv"):
    df = pd.read_csv("scraped_news_v9.csv")
    print(f"📊 Total: {len(df)}")
    display(df.tail())
else: print("No data.")